# 📘 부록 01: vLLM 서버 로그와 처리량 읽기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<div style="background:#ffffff;color:#202124;font-size:17px;line-height:1.6;padding:6px 10px;border-left:4px solid #54769b;">
<strong style="color:#1f3a5f;font-size:19px;">실습 환경</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">읽기 자료:</strong> 노트북에서 실행할 셀이 없습니다. 예시 서버 로그에서 뽑은 줄을 읽습니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">내 서버 로그:</strong> 5절의 명령만 RunPod 터미널 3에서 실행해 바로 봅니다.
</div>
</div>

#### **서버 통계 로그**

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM 서버가 <strong style="font-size:1.13em;color:#1f3a5f;">기본 약 10초마다 남기는 작업 현황</strong>입니다. 서버 전체의 입력·출력 처리량과 현재 처리 중이거나 기다리는 요청 수를 보여 줍니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01의 동시 요청 벤치마크를 마친 뒤 읽습니다. 노무 상담 서버의 실제 로그를 보고, 내 답변의 속도와 서버 전체 처리량을 구분합니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

**학습 목표**

1. 서버 통계 로그 한 줄의 항목을 설명합니다.
2. 입력·출력 처리량과 TPOT를 숫자 예시로 설명합니다.
3. 요청 하나의 생성 속도와 서버 전체 처리량을 구분합니다.
4. 내 서버의 통계 줄을 RunPod 터미널에서 바로 봅니다.

</div>

## 1. 로그 한 줄의 항목

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 로그는 서버가 입력을 초당 904.5토큰 계산하고, 출력을 초당 408.8토큰 만든 구간입니다. 로그를 찍는 순간에는 요청 16개를 처리 중입니다.</p>
</div>

```text
187: (APIServer pid=34193) INFO 10-30 08:01:46 [loggers.py:335] Engine 000: Avg prompt throughput: 904.5 tokens/s, Avg generation throughput: 408.8 tokens/s, Running: 16 reqs, Waiting: 0 reqs, GPU KV cache usage: 10.8%, Prefix cache hit rate: 93.1%
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">줄 앞의 숫자(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">187:</code>):</strong> 예시 로그 파일의 줄 번호입니다. 내 터미널 화면에는 찍히지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">버전:</strong> 예시는 vLLM 0.32.0으로 찍은 로그입니다. 수업 버전 0.27.0도 처리량 계산 방식은 같고, 화면에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[loggers.py:310]</code>처럼 괄호 안 번호만 다르게 찍힙니다.</li>
</ul>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐싱(prefix caching)</strong>은 입력의 같은 앞부분을 다시 계산하지 않고 저장된 계산 결과를 재사용하는 기능입니다. 예를 들어 여러 질문의 시스템 메시지가 같으면 그 부분의 계산을 줄일 수 있습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">뜻</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Avg prompt throughput</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">입력 처리량. 구간 동안 새로 계산한 것으로 집계된 입력 토큰의 초당 개수</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Avg generation throughput</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량. 구간 동안 모든 요청에 만든 출력 토큰의 초당 개수</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">줄을 찍는 순간 함께 처리 중인 요청 수. 구간 평균이 아닙니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">줄을 찍는 순간 차례를 기다리는 요청 수</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU KV cache usage</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">KV 캐시 블록 가운데 현재 요청이 사용하는 비율. 전체 GPU 메모리 사용률과 다릅니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Prefix cache hit rate</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">최근 요청 약 1,000개의 입력 토큰 가운데 접두어 캐시에서 찾은 비율</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting: 43</code>이면 요청 43개가 차례를 기다리고 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting</code>은 구간 평균이 아니라 로그를 찍는 순간의 값입니다.</li>
</ul>
</div>

## 2. 입력 처리량과 출력 처리량

### 2-1. 구간 길이

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">구간</strong>은 이전 통계 집계부터 이번 집계까지의 시간입니다. 기본 주기는 10초지만, 처리량은 <strong style="font-size:1.13em;color:#1f3a5f;">실제로 흐른 시간</strong>으로 나눕니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예:</strong> 실제 구간이 10.2초이면 10이 아니라 10.2로 나눕니다. 첫 집계 구간은 더 길 수도 있습니다.</li>
<li style="margin:3px 0;">주기는 서버를 시작할 때 환경변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_LOG_STATS_INTERVAL</code>로 바꿀 수 있습니다.</li>
</ul>
</div>

### 2-2. 입력 처리량

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">구간 동안 <strong style="font-size:1.13em;color:#1f3a5f;">prefill(입력 처리)로 새로 계산한 입력 토큰</strong>의 초당 개수입니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">입력 처리량</strong> = 구간에 집계된 새로 계산한 입력 토큰 수 ÷ 실제 구간 길이(초)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">세는 토큰:</strong> 질문의 입력 토큰만 셉니다. 캐시에서 다시 쓴 토큰은 빼고, decode에서 방금 만든 토큰을 다시 넣어 계산하는 것도 세지 않습니다. 그 토큰은 출력 처리량에서 셉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">나누는 시간:</strong> 구간 전체입니다. 그동안 다른 요청의 decode를 한 시간과 쉰 시간도 들어갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">계산 예:</strong> 10초 구간에 입력 1,000토큰 중 800토큰은 캐시에서 다시 쓰고 200토큰을 0.05초 만에 계산했습니다. 입력 처리량은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">200 ÷ 10 = 20토큰/초</code>입니다. prefill 계산 속도(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">200 ÷ 0.05 = 4,000토큰/초</code>)가 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">여러 단계로 나누는 계산:</strong> 한 단계에 계산할 토큰 수에는 상한(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-num-batched-tokens</code>, 24GB GPU 기본 2,048)이 있어, 긴 입력은 여러 단계로 나눠 prefill합니다. 예: 캐시 적중 없이 5,000토큰이면 2,048 + 2,048 + 904토큰의 세 단계입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">집계 시점:</strong> 나눠 계산해도 첫 출력 토큰이 나오는 단계에서 입력 토큰 수를 한 번에 더합니다. 위 예에서는 셋째 단계가 속한 구간에 5,000토큰이 전부 잡힙니다.</li>
</ul>
</div>

### 2-3. 출력 처리량

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버가 <strong style="font-size:1.13em;color:#1f3a5f;">모든 요청에 만든 출력 토큰을 합친 초당 개수</strong>입니다. 첫 출력 토큰도 포함합니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">출력 처리량</strong> = 구간 동안 모든 요청에 만든 출력 토큰 수의 합 ÷ 실제 구간 길이(초)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">계산 예:</strong> 10초 동안 질문 A의 답변 300토큰, 질문 B의 답변 200토큰을 만들었다면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">(300 + 200) ÷ 10 = 50토큰/초</code>입니다.</li>
</ul>
</div>

### 2-4. TPOT와 요청 하나의 생성 속도

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TTFT(Time To First Token, 첫 토큰 지연)</strong>는 질문을 보낸 뒤 첫 출력 토큰을 받을 때까지의 시간입니다.
<strong style="font-size:1.13em;color:#1f3a5f;">TPOT(Time Per Output Token, 출력 토큰당 시간)</strong>는 첫 토큰 이후 토큰 하나가 추가되는 데 걸린 평균 시간입니다. 단위는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ms/토큰</code>이며 작을수록 답변이 빠르게 이어집니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">질문 하나의 응답 예</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">시간</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">질문 전송 → 첫 출력 토큰</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">500ms → TTFT</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">첫 토큰 → 두 번째 토큰</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">20ms</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">두 번째 토큰 → 세 번째 토큰</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">20ms</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">첫 토큰 이후 간격의 평균</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">20ms/토큰 → TPOT</td>
</tr>
</tbody>
</table>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">TPOT</strong> = 첫 토큰부터 마지막 토큰까지 걸린 시간(ms) ÷ (출력 토큰 수 − 1)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">출력 3토큰 사이의 간격은 2개이므로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">40 ÷ (3 − 1) = 20ms/토큰</code>입니다. <strong style="font-size:1.13em;color:#1f3a5f;">요청 하나의 생성 속도</strong>는 이 답변이 첫 토큰 이후 초당 몇 토큰씩 이어지는지를 나타냅니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">요청 하나의 생성 속도</strong> = 1,000 ÷ TPOT(ms/토큰)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">TPOT가 20ms/토큰이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">1,000 ÷ 20 = 50토큰/초</code>입니다. 1초가 1,000ms이므로 1,000을 TPOT로 나눕니다. TTFT는 이 속도에 포함하지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비교 예:</strong> 요청 3개가 같은 1초 동안 각각 50토큰씩 만든다면, 요청 하나는 50토큰/초이고 서버 전체는 150토큰/초입니다. 실제로는 동시 요청을 늘리면 각 요청의 TPOT도 달라지므로 처리량이 요청 수에 비례하지는 않습니다.</p>
</div>

## 3. 요청 하나일 때 숫자가 작은 이유

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래는 요청 하나를 보낸 뒤의 로그입니다.</p>
</div>

```text
142: INFO:     127.0.0.1:52114 - "POST /v1/chat/completions HTTP/1.1" 200 OK
143: (APIServer pid=34193) INFO 10-30 08:01:26 [loggers.py:335] Engine 000: Avg prompt throughput: 432.6 tokens/s, Avg generation throughput: 22.2 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
144: (APIServer pid=34193) INFO 10-30 08:01:36 [loggers.py:335] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 요청은 입력 4,979토큰, 출력 256토큰이었습니다. 첫 집계 구간은 약 11.5초였으므로 출력 처리량이 다음처럼 나옵니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">출력 처리량</strong> = 256 ÷ 약 11.5초 = 약 22토큰/초</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">142번 줄(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">POST ... 200 OK</code>):</strong> 요청을 받아 처리했다는 접근 기록입니다. 토큰 수나 속도는 없습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">22.2:</strong> 요청이 일한 시간에 더해 아무 일도 하지 않은 시간까지 포함한 구간 평균입니다. 이 요청의 TPOT를 환산한 속도와 다를 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Running 0:</strong> 로그를 찍는 순간에는 요청이 이미 끝났습니다. 그 구간에 일을 하지 않았다는 뜻은 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">다음 줄의 0.0:</strong> 다음 구간에는 만든 토큰이 없습니다. 계속 일이 없으면 이후의 0.0 줄은 기본 로그에서 생략됩니다.</li>
</ul>
</div>

## 4. 동시 요청이 많을 때 달라지는 값

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버는 여러 요청을 함께 처리할 수 있습니다. 요청마다 답변이 조금 느려져도, 여러 답변의 출력을 합친 서버 전체 처리량은 커질 수 있습니다.</p>
</div>

```text
226: (APIServer pid=34193) INFO 10-30 08:02:26 [loggers.py:335] Engine 000: Avg prompt throughput: 111.7 tokens/s, Avg generation throughput: 63.8 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.9%, Prefix cache hit rate: 94.1%
187: (APIServer pid=34193) INFO 10-30 08:01:46 [loggers.py:335] Engine 000: Avg prompt throughput: 904.5 tokens/s, Avg generation throughput: 408.8 tokens/s, Running: 16 reqs, Waiting: 0 reqs, GPU KV cache usage: 10.8%, Prefix cache hit rate: 93.1%
478: (APIServer pid=34193) INFO 10-30 08:04:16 [loggers.py:335] Engine 000: Avg prompt throughput: 158.4 tokens/s, Avg generation throughput: 5.7 tokens/s, Running: 8 reqs, Waiting: 43 reqs, GPU KV cache usage: 6.5%, Prefix cache hit rate: 95.1%
592: (APIServer pid=34193) INFO 10-30 08:04:46 [loggers.py:335] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 362.6 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 95.2%
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">어느 측정의 줄인가:</strong> 226번은 동시 요청 1, 187번은 동시 요청 16, 478번과 592번은 동시 요청 64를 측정하던 구간입니다. 478번은 64번 측정의 시작, 592번은 끝입니다.</li>
</ul>
<p style="margin:6px 0;">아래는 교안 01과 같은 노무 입력으로 측정한 벤치마크 결과입니다. <strong style="font-size:1.13em;color:#1f3a5f;">로그는 약 10초 단위, 벤치마크는 측정 시작부터 끝까지</strong>를 기준으로 평균을 내므로 처리량 숫자가 다를 수 있습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">동시 요청 수</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량(토큰/초)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">평균 TPOT(ms)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">평균 TTFT(ms)</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">63.8</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">15.3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">56.5</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">16</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">611.3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">21.6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">184.2</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">64</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1,064.1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">45.8</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1,212.9</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 전체:</strong> 동시 요청 1 → 64에서 출력 처리량은 63.8 → 1,064.1토큰/초로 커졌습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 하나:</strong> 같은 비교에서 TPOT는 15.3 → 45.8ms로 늘어, 각 답변은 더 느리게 이어집니다. TTFT도 길어집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">로그 478번 줄:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting: 43</code>입니다. 동시 요청을 늘리다가 대기가 쌓이면 TTFT와 TPOT도 함께 보고 허용할 요청 수를 정합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">로그 592번 줄:</strong> 입력은 0.0인데 출력은 362.6입니다. 이 구간에는 새 요청의 입력 집계 없이, 이미 처리 중이던 요청의 답변만 마저 만들었습니다.</li>
</ul>
</div>

## 5. 내 서버의 통계 줄 보기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01 5-1의 터미널 3과 같은 명령입니다. VS Code 원격 창에서 새 터미널 3을 열고 아래 명령을 켜 둔 채, 터미널 2에서 벤치마크를 실행하면 통계 줄이 계속 나타납니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 3**

```bash
# 실습자료 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다

# 서버 로그에 새로 찍히는 처리량 줄만 바로바로 보여 줍니다(멈출 때까지 계속 돕니다)
tail -f output/qwen3.5-4b_server.log | grep --line-buffered "throughput"
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">명령어 설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정 의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tail -f output/qwen3.5-4b_server.log</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">로그 파일 끝에 새로 붙는 줄을 계속 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">grep --line-buffered &quot;throughput&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">그 가운데 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">throughput</code>이 든 줄만, 모아 두지 않고 한 줄씩 바로 내보냅니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">다 봤으면 터미널 3에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 멈춥니다. 로그에서는 서버 전체 처리량을, 벤치마크 결과의 TPOT에서는 요청별 답변이 이어지는 빠르기를 확인합니다.</p>
</div>

## 6. 핵심 정리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">구간:</strong> 기본 약 10초이며, 처리량은 실제로 흐른 시간으로 나눕니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력·출력:</strong> 입력은 새로 계산한 토큰을, 출력은 모든 요청에 만든 토큰을 셉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TPOT:</strong> 첫 토큰 이후 토큰 하나에 걸린 평균 시간입니다. 20ms/토큰이면 초당 50토큰입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">로그 해석:</strong> 처리량은 구간 평균, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting</code>은 순간값입니다. 서버 전체 처리량이 커져도 각 답변은 느려질 수 있습니다.</li>
</ul>
</div>